# Exact quarter-ring with NURBS

This notebook builds an **exact** quarter-ring (annular sector, 0° to 90°) using
**Non-Uniform Rational B-Splines (NURBS)**.

| Parameter | Value |
|---|---|
| Inner radius $r_1$ | 1.0 |
| Outer radius $r_2$ | 2.0 |
| Angular span | $0$ to $\pi/2$ |

### Why NURBS?

A circular arc is a conic section and **cannot be represented exactly** by a polynomial
(B-spline) basis.  The rational basis of NURBS adds one weight $w_a$ per control point,
replacing the B-spline basis $N_a$ by:
$$
R_a(\xi) = \frac{w_a\, N_a(\xi)}{\displaystyle\sum_b w_b\, N_b(\xi)}
$$
With the right weight on the corner control point ($w = 1/\sqrt{2}$),
the degree-2 NURBS curve through three control points traces an **exact** quarter-circle.
All other weights are 1 (no rational correction needed for straight edges).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager,
    Patch, HRefiner, SubdivisionRefiner, PRefiner, BezierExtractor,
)
from yeti_iga.future.plotting import plot_patches_2d

## Control-point network

The quarter-ring is parameterised as a $3 \times 2$ NURBS patch:

- **$u$ direction** — angular, degree 2, knot vector `[0,0,0,1,1,1]`
- **$v$ direction** — radial, degree 1, knot vector `[0,0,1,1]`

```
iv=1 (outer)   (r2,0) w=1 ── (r2,r2) w=1/√2 ── (0,r2) w=1
iv=0 (inner)   (r1,0) w=1 ── (r1,r1) w=1/√2 ── (0,r1) w=1
               iu=0             iu=1               iu=2
```

The weight $w = 1/\sqrt{2}$ on each corner point is the classical NURBS value that
makes the mid-arc point lie **exactly** on a circle.

In [ ]:
r1, r2 = 1.0, 2.0
w_corner = 1.0 / np.sqrt(2.0)   # NURBS weight for quarter-circle corner CP

# --- Control-point manager (NURBS: corner points have w ≠ 1) ---
mgr = ControlPointManager(dim=2)
# Inner arc (iv = 0)
mgr.add_point([r1,  0.0], w=1.0     )   # 0 – start
mgr.add_point([r1,  r1 ], w=w_corner)   # 1 – corner (weight activates NURBS mode)
mgr.add_point([0.0, r1 ], w=1.0     )   # 2 – end
# Outer arc (iv = 1)
mgr.add_point([r2,  0.0], w=1.0     )   # 3 – start
mgr.add_point([r2,  r2 ], w=w_corner)   # 4 – corner
mgr.add_point([0.0, r2 ], w=1.0     )   # 5 – end

print(f'is_rational : {mgr.is_rational}')
print(f'weights     : {mgr.weights_view()}')

## Build the NURBS patch

`plot_patches_2d` handles NURBS transparently: iso-parametric lines are evaluated
via `evaluate_patch_nd_omp`, which applies the rationalization $R_a = w_a N_a / W$
automatically when `cp_manager.is_rational` is `True`.

With `show_weights=True`, control points whose weight differs from 1 are drawn as
**circles** (instead of squares) and annotated with their weight value.

In [ ]:
su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))  # degree 2 – angular
sv = BSpline(1, np.array([0., 0., 1., 1.]))            # degree 1 – radial

mapping     = [0, 1, 2, 3, 4, 5]   # u-fastest flat order
local_shape = [3, 2]

patch = Patch(BSplineSurface(su, sv), mgr, mapping, local_shape)

plot_patches_2d(patch,
                show_control_points=True,
                show_control_point_indices=True,
                show_weights=True,
                title='Exact quarter-ring NURBS — initial mesh (1 element)')

## Geometry verification — exact circular arcs

Unlike a pure B-spline approximation, the NURBS mapping gives **zero radial error**
along the inner and outer arcs.

In [ ]:
n_sample = 300
u_vals   = np.linspace(0.0, 1.0, n_sample)

def arc_points(patch, v_fixed):
    su, sv = patch.tensor.components
    spans  = np.array([[su.find_span(u), sv.find_span(v_fixed)] for u in u_vals],
                      dtype=np.int32)
    params = np.column_stack([u_vals, np.full(n_sample, v_fixed)])
    return patch.evaluate_patch_nd_omp(spans, params)

inner_pts = arc_points(patch, v_fixed=0.0)
outer_pts = arc_points(patch, v_fixed=1.0)
inner_r   = np.linalg.norm(inner_pts, axis=1)
outer_r   = np.linalg.norm(outer_pts, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: overlay on exact arcs
ax = axes[0]
theta = np.linspace(0, np.pi / 2, n_sample)
ax.plot(r1 * np.cos(theta), r1 * np.sin(theta), 'k-',  lw=3,   label='exact', zorder=1)
ax.plot(r2 * np.cos(theta), r2 * np.sin(theta), 'k-',  lw=3,   zorder=1)
ax.plot(inner_pts[:, 0], inner_pts[:, 1], 'b--', lw=1.5, label='NURBS inner', zorder=2)
ax.plot(outer_pts[:, 0], outer_pts[:, 1], 'r--', lw=1.5, label='NURBS outer', zorder=2)
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.set_title('NURBS arcs overlaid on exact circles')
ax.grid(True, alpha=0.4)

# Right: radial error (should be machine-precision zero)
ax = axes[1]
ax.semilogy(u_vals, np.abs(inner_r - r1) + 1e-17, 'b-', lw=2, label=f'inner (r₁={r1})')
ax.semilogy(u_vals, np.abs(outer_r - r2) + 1e-17, 'r-', lw=2, label=f'outer (r₂={r2})')
ax.set_xlabel('u')
ax.set_ylabel('|radial error|')
ax.set_title('Radial error — NURBS is exact to machine precision')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print(f'Max inner radial error: {np.abs(inner_r - r1).max():.2e}')
print(f'Max outer radial error: {np.abs(outer_r - r2).max():.2e}')

## NURBS refinement

The three refinement operators (`HRefiner`, `PRefiner`, `SubdivisionRefiner`) have been
extended to handle NURBS: blending is performed in **homogeneous coordinates**
$(w\cdot x,\; w\cdot y,\; w)$ so that the rational geometry is preserved exactly after
every refinement step.

Here we apply **$k$-refinement** (degree elevation then subdivision):

1. `PRefiner(0, 1)` — raise the angular degree from 2 to 3
2. `SubdivisionRefiner(0, 2)` — bisect angular spans 2× → 4 angular elements
3. `SubdivisionRefiner(1, 1)` — bisect radial span → 2 radial elements

In [ ]:
PRefiner(direction=0, n_elevations=1).refine(patch)       # degree 2 → 3
SubdivisionRefiner(direction=0, n_levels=2).refine(patch)  # 4 angular elements
SubdivisionRefiner(direction=1, n_levels=1).refine(patch)  # 2 radial elements

su_ref, sv_ref = patch.tensor.components
print(f'Degree after k-refinement : p_u={su_ref.degree}, p_v={sv_ref.degree}')
print(f'Knot vector u             : {su_ref.knot_vector}')
print(f'Control points            : {mgr.n_points}')
print(f'Still rational            : {mgr.is_rational}')

plot_patches_2d(patch,
                show_control_points=True,
                show_weights=True,
                title=f'After k-refinement  '
                      f'(degree {su_ref.degree}×{sv_ref.degree}, '
                      f'{mgr.n_points} CPs)')

## Geometry still exact after refinement

Because refinement operates in homogeneous coordinates, the NURBS geometry is preserved.

In [ ]:
inner_pts_ref = arc_points(patch, v_fixed=0.0)
outer_pts_ref = arc_points(patch, v_fixed=1.0)

print(f'Max inner radial error after refinement: {np.abs(np.linalg.norm(inner_pts_ref, axis=1) - r1).max():.2e}')
print(f'Max outer radial error after refinement: {np.abs(np.linalg.norm(outer_pts_ref, axis=1) - r2).max():.2e}')

## Bézier extraction

In [ ]:
elems = BezierExtractor.extract_nd(patch)
pu, pv = patch.tensor.components[0].degree, patch.tensor.components[1].degree
print(f'{len(elems)} Bézier elements, degree {pu}×{pv}')

rational    = patch.cp_manager.is_rational
all_weights = patch.cp_manager.weights_view() if rational else None

def bezier_cps(elem):
    """Physical Bézier CPs, correctly computed in homogeneous space for NURBS."""
    active   = list(elem.active_indices)
    P_active = np.array([patch.control_point(j) for j in active])
    if rational:
        w = all_weights[active]
        w_bz = elem.C.T @ w
        P_bz = (elem.C.T @ (w[:, None] * P_active)) / w_bz[:, None]
    else:
        P_bz = elem.C.T @ P_active
    return P_bz

u_breaks = np.unique(patch.tensor.components[0].knot_vector)
v_breaks = np.unique(patch.tensor.components[1].knot_vector)
colors   = plt.cm.tab10(np.arange(len(elems)) % 10)

fig, ax = plt.subplots(figsize=(6, 6))

su, sv = patch.tensor.components
n_samp = 80
for u_val in u_breaks:
    vs  = np.linspace(v_breaks[0], v_breaks[-1], n_samp)
    sp  = np.array([[su.find_span(u_val), sv.find_span(v)] for v in vs], dtype=np.int32)
    pts = patch.evaluate_patch_nd_omp(sp, np.column_stack([np.full(n_samp, u_val), vs]))
    ax.plot(pts[:, 0], pts[:, 1], 'b-', lw=1)
for v_val in v_breaks:
    us  = np.linspace(u_breaks[0], u_breaks[-1], n_samp)
    sp  = np.array([[su.find_span(u), sv.find_span(v_val)] for u in us], dtype=np.int32)
    pts = patch.evaluate_patch_nd_omp(sp, np.column_stack([us, np.full(n_samp, v_val)]))
    ax.plot(pts[:, 0], pts[:, 1], 'b-', lw=1)

for elem, color in zip(elems, colors):
    P_bz   = bezier_cps(elem)
    P_grid = P_bz.reshape(pv + 1, pu + 1, 2)
    for iv in range(pv + 1):
        ax.plot(P_grid[iv, :, 0], P_grid[iv, :, 1], '--', color=color, lw=0.9)
    for iu in range(pu + 1):
        ax.plot(P_grid[:, iu, 0], P_grid[:, iu, 1], '--', color=color, lw=0.9)
    ax.scatter(P_bz[:, 0], P_bz[:, 1], s=25, c=[color], zorder=3)
    cx, cy = P_bz.mean(axis=0)
    ax.annotate(f'e{tuple(elem.elem_index)}', (cx, cy),
                ha='center', va='center', fontsize=8, color=color, fontweight='bold')

theta = np.linspace(0, np.pi / 2, 300)
ax.plot(r1 * np.cos(theta), r1 * np.sin(theta), 'k:', lw=1.5, label='exact arcs')
ax.plot(r2 * np.cos(theta), r2 * np.sin(theta), 'k:', lw=1.5)

ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.set_title(f'Bézier elements (degree {pu}×{pv}) — NURBS homogeneous blending')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## VTU export

In [ ]:
from yeti_iga.future.vtu import write_bezier_patch_vtu
import os

os.makedirs('output', exist_ok=True)

# Compute the "radius" field correctly at each B-spline control point.
#
# norm(CP_position) is WRONG: the corner CP (r,r) with weight 1/√2 has
# norm r√2, not r.  VTK then interpolates the field with the rational
# Bézier formula, so to get concentric color bands we must provide the
# PHYSICAL RADIUS at the Greville abscissa of each CP in the v-direction.
#
# For pv=1 with uniform v-weights:  radius(u,v) = r1 + v*(r2-r1).
# Setting f_iv = r1 + GA_v[iv]*(r2-r1) makes VTK's rational interpolation
# reproduce this exactly on the surface.

sv_spline = patch.tensor.components[1]
pv_deg    = sv_spline.degree
kv_v      = np.array(sv_spline.knot_vector)
n_u       = patch.local_shape[0]
n_v       = patch.local_shape[1]

# Greville abscissae of the v-spline: GA_iv = mean(kv[iv+1 : iv+pv+1])
ga_v = np.array([np.mean(kv_v[iv + 1 : iv + pv_deg + 1]) for iv in range(n_v)])

# u-fastest flat ordering: flat = iu + iv*n_u  →  iv = flat // n_u
radius = np.array([r1 + ga_v[i // n_u] * (r2 - r1) for i in range(patch.n_cp)])

write_bezier_patch_vtu(patch, 'output/quarter_ring_nurbs.vtu',
                        field=radius, field_name='radius')
print('Exported → output/quarter_ring_nurbs.vtu')
print(f'Field range: [{radius.min():.3f}, {radius.max():.3f}]  (expected [{r1}, {r2}])')

## Stiffness matrix (steel: E = 210 000, ν = 0.3)

`PatchIntegrator` assembles the stiffness matrix by Gauss quadrature using the
**rationalized NURBS basis** ($R_a = w_a N_a / W$) instead of the raw B-spline basis.

Because `dof_manager` is read-only on `Patch`, we reconstruct the patch carrying a fresh
`PatchDOFManager` (2 degrees of freedom per control point: $u_x$, $u_y$).


In [ ]:
from yeti_iga.future.bspline import (
    GlobalDOFManager, PatchDOFManager, PatchIntegrator, MaterialProperties, IGABasis1D,
)

# Reconstruct the patch with a DOF manager (2 DOF per control point: ux, uy).
# dof_manager is read-only on Patch, so we wrap the same tensor/cp_manager
# in a new Patch object that carries the DOF manager.
n_cp    = patch.n_cp
n_dof   = 2 * n_cp          # 2D problem: ux + uy per control point
gm      = GlobalDOFManager([2] * n_cp)
pdm     = PatchDOFManager(2, list(range(n_cp)), gm)
patch_k = Patch(patch.tensor, patch.cp_manager,
                list(patch.global_indices), list(patch.local_shape), pdm)

# Gauss quadrature: degree+1 points per span per direction (exact for bilinear forms).
su_k, sv_k = patch_k.tensor.components
basis_u = IGABasis1D.build(su_k, su_k.degree + 1)
basis_v = IGABasis1D.build(sv_k, sv_k.degree + 1)

# Steel material (plane stress / 2-D solid)
steel = MaterialProperties(E=210_000.0, nu=0.3)

K = PatchIntegrator(patch_k, basis_u, basis_v, steel).integrate_stiffness()

K_arr   = K.toarray()
sym_err = np.max(np.abs(K_arr - K_arr.T))

print(f'Control points : {n_cp}')
print(f'DOF            : {n_dof}')
print(f'K shape        : {K.shape}')
print(f'K non-zeros    : {K.nnz}')
print(f'Sparsity       : {1 - K.nnz / n_dof**2:.1%}')
print(f'Symmetry error : {sym_err:.2e}  (should be < 1e-10)')

# Sparsity pattern
fig, ax = plt.subplots(figsize=(5, 5))
ax.spy(K_arr != 0, markersize=2)
ax.set_title(f'Sparsity pattern — K ({n_dof}×{n_dof}, {K.nnz} nnz)')
plt.tight_layout()
plt.show()
